## Steps involved:

1. Data Ingestion : WebBaseLoader
2. Data Transformation : RecursiveCharacterTextSplitter
3. Embeddings and VectorDB
4. Create Stuff-Documents Chain using : LLM and ChatPromptTemplate.
5. Create "Retrieval Chain" using the "vector_db.as_retriever()" and "stuff_documents_chain"
6. Invoke the Retrieval Chain : Using "input" as field. (Context will be auto-provided using this Retrieval Chain)

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()


os.environ['GOOGLE_API_KEY'] = os.getenv('GOOGLE_API_KEY')
os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGCHAIN_API_KEY') ## for LangSmith Tracking
os.environ['LANGSMITH_TRACING_V2'] = "true"
os.environ['LANGCHAIN_PROJECT'] = os.getenv('LANGCHAIN_PROJECT')

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
  model = "gemini-2.5-flash"
)
print(llm)

model='models/gemini-2.5-flash' google_api_key=SecretStr('**********') client=<google.ai.generativelanguage_v1beta.services.generative_service.client.GenerativeServiceClient object at 0x000001F69BB4A8D0> default_metadata=() model_kwargs={}


# Data Ingestion

In [3]:
from langchain_community.document_loaders import WebBaseLoader

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [22]:
loader = WebBaseLoader(
  web_path= "https://python.langchain.com/docs/introduction/"
)
loader

In [23]:
docs = loader.load()
docs

[Document(metadata={'source': 'https://python.langchain.com/docs/introduction/', 'title': 'Introduction | 🦜️🔗 LangChain', 'description': 'LangChain is a framework for developing applications powered by large language models (LLMs).', 'language': 'en'}, page_content='\n\n\n\n\nIntroduction | 🦜️🔗 LangChain\n\n\n\n\n\n\n\n\nSkip to main contentThese docs will be deprecated and no longer maintained with the release of LangChain v1.0 in October 2025. Visit the v1.0 alpha docsIntegrationsAPI ReferenceMoreContributingPeopleError referenceLangSmithLangGraphLangChain HubLangChain JS/TSv0.3v0.3v0.2v0.1💬SearchIntroductionTutorialsBuild a Question Answering application over a Graph DatabaseTutorialsBuild a simple LLM application with chat models and prompt templatesBuild a ChatbotBuild a Retrieval Augmented Generation (RAG) App: Part 2Build an Extraction ChainBuild an AgentTaggingBuild a Retrieval Augmented Generation (RAG) App: Part 1Build a semantic search engineBuild a Question/Answering system

# Data Transformation : Text Splitting

In [24]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
  chunk_size = 1000,
  chunk_overlap = 200
)
splitted_docs = text_splitter.split_documents(documents= docs)
splitted_docs

[Document(metadata={'source': 'https://python.langchain.com/docs/introduction/', 'title': 'Introduction | 🦜️🔗 LangChain', 'description': 'LangChain is a framework for developing applications powered by large language models (LLMs).', 'language': 'en'}, page_content='Introduction | 🦜️🔗 LangChain'),
 Document(metadata={'source': 'https://python.langchain.com/docs/introduction/', 'title': 'Introduction | 🦜️🔗 LangChain', 'description': 'LangChain is a framework for developing applications powered by large language models (LLMs).', 'language': 'en'}, page_content='Skip to main contentThese docs will be deprecated and no longer maintained with the release of LangChain v1.0 in October 2025. Visit the v1.0 alpha docsIntegrationsAPI ReferenceMoreContributingPeopleError referenceLangSmithLangGraphLangChain HubLangChain JS/TSv0.3v0.3v0.2v0.1💬SearchIntroductionTutorialsBuild a Question Answering application over a Graph DatabaseTutorialsBuild a simple LLM application with chat models and prompt te

In [29]:
print(f"Total chunks : {len(splitted_docs)}")

Total chunks : 18


# Embedding and Vector DB

In [26]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
  model="models/gemini-embedding-001"
)

In [31]:
from langchain_community.vectorstores import FAISS

db = FAISS.from_documents(
  documents= splitted_docs,  ## make embeddings and store in vector DB
  embedding= embeddings
)

db

In [34]:
query = "Build your applications using LangChain's open-source components"
result = db.similarity_search(query)
result[0].page_content

'LangChain is a framework for developing applications powered by large language models (LLMs).\nLangChain simplifies every stage of the LLM application lifecycle:'

### So this actually just gives resutls similar to the query we have asked based on similarity of vectors near to the query vector. But we need to add retreival chain for making it meaningful.

### We use : "create_stuff_documents_chain"

create_stuff_documents_chain is a LangChain helper that builds a simple chain for RAG (Retrieval-Augmented Generation) when you want to “stuff” all retrieved documents directly into the LLM prompt

📌 What does "stuff" mean here?

There are different ways to pass retrieved docs to an LLM:

Stuffing → put all docs together in the prompt (simple, but can hit token limits).

Map-reduce → summarize each doc, then combine summaries.

Refine → iteratively refine an answer by going through docs one by one.

create_stuff_documents_chain gives you the stuffing strategy.

#### Using this Stuff Documents chain, we have : PROMPT + LLM + OUTPUT_PARSER, all three conmbined with each other in a single chain.

In [ ]:
## Retieval Chain, Document Chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
  """
    Answer the following questions based only on the provided context:
    <context>
    {context} 
    </context>
  """
)

document_chain = create_stuff_documents_chain(
  llm= llm,
  prompt= prompt
)

document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n    Answer the following questions based only on the provided context:\n    <context>\n    {context} \n    </conte>xt\n  '), additional_kwargs={})])
| ChatGoogleGenerativeAI(model='models/gemini-2.5-flash', google_api_key=SecretStr('**********'), client=<google.ai.generativelanguage_v1beta.services.generative_service.client.GenerativeServiceClient object at 0x000001F69BB4A8D0>, default_metadata=(), model_kwargs={})
| StrOutputParser(), kwargs={}, config={'run_name': 'stuff_documents_chain'}, config_factories=[])

## Testing stuff_documents_chain:

In [41]:
from langchain_core.documents import Document
print(document_chain.invoke(
  { "input" : "The LangChain framework consists of multiple open-source libraries",
    "context" : [Document(page_content=
       """The LangChain framework consists of multiple open-source libraries. Read more in the Architecture page.

          langchain-core: Base abstractions for chat models and other components.
          Integration packages (e.g. langchain-openai, langchain-anthropic, etc.): Important integrations have been split into lightweight packages that are co-maintained by the LangChain team and the integration developers.
          langchain: Chains, agents, and retrieval strategies that make up an application's cognitive architecture.
          langchain-community: Third-party integrations that are community maintained.
          langgraph: Orchestration framework for combining LangChain components into production-ready applications with persistence, streaming, and other key features. See LangGraph documentation.""")]
  }
))

Based on the provided context:

*   **langchain-core:** Provides base abstractions for chat models and other components.
*   **Integration packages (e.g., langchain-openai, langchain-anthropic):** Are lightweight packages for important integrations, co-maintained by the LangChain team and integration developers.
*   **langchain:** Contains chains, agents, and retrieval strategies that make up an application's cognitive architecture.
*   **langchain-community:** Consists of third-party integrations that are community maintained.
*   **langgraph:** Is an orchestration framework for combining LangChain components into production-ready applications, offering features like persistence and streaming.
*   More information about the LangChain framework's architecture can be found on the **Architecture page**.
*   More information about **LangGraph** can be found in its documentation.


## Create "Retrieval Chain" using the "vector_db.as_retriever()" and "stuff_documents_chain":

In [45]:
retriever = db.as_retriever()

from langchain.chains import create_retrieval_chain

retrieval_chain = create_retrieval_chain(retriever, document_chain)
retrieval_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001F6BB395050>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n    Answer the following questions based only on the provided context:\n    <context>\n    {context} \n    </conte>xt\n  '), additional

In [46]:
## Get response
response = retrieval_chain.invoke({"input" : "The LangChain framework consists of multiple open-source libraries."})

In [49]:
print(response)

{'input': 'The LangChain framework consists of multiple open-source libraries.', 'context': [Document(id='b9a8ac52-bbf9-47c9-a883-b81ca093286b', metadata={'source': 'https://python.langchain.com/docs/introduction/', 'title': 'Introduction | 🦜️🔗 LangChain', 'description': 'LangChain is a framework for developing applications powered by large language models (LLMs).', 'language': 'en'}, page_content='LangChain is a framework for developing applications powered by large language models (LLMs).\nLangChain simplifies every stage of the LLM application lifecycle:'), Document(id='b056df70-8b39-4072-811b-23a9dfe89df2', metadata={'source': 'https://python.langchain.com/docs/introduction/', 'title': 'Introduction | 🦜️🔗 LangChain', 'description': 'LangChain is a framework for developing applications powered by large language models (LLMs).', 'language': 'en'}, page_content="langchain-core: Base abstractions for chat models and other components.\nIntegration packages (e.g. langchain-openai, langch

In [48]:
print(response["answer"])

Based on the provided context:

1.  **What is LangChain?**
    LangChain is a framework for developing applications powered by large language models (LLMs).

2.  **What is the purpose of `langchain-core`?**
    `langchain-core` provides base abstractions for chat models and other components.

3.  **What is the difference between `langchain` and `langchain-community`?**
    `langchain` provides chains, agents, and retrieval strategies that make up an application's cognitive architecture. `langchain-community` provides third-party integrations that are community maintained.

4.  **What is `langgraph` used for?**
    `langgraph` is an orchestration framework for combining LangChain components into production-ready applications with persistence, streaming, and other key features. It is also used to build stateful agents with first-class streaming and human-in-the-loop support.

5.  **Which section is recommended for hands-on learners to get started?**
    The Tutorials section is recommend

In [ ]:
print(response["context"])

[Document(id='b9a8ac52-bbf9-47c9-a883-b81ca093286b', metadata={'source': 'https://python.langchain.com/docs/introduction/', 'title': 'Introduction | 🦜️🔗 LangChain', 'description': 'LangChain is a framework for developing applications powered by large language models (LLMs).', 'language': 'en'}, page_content='LangChain is a framework for developing applications powered by large language models (LLMs).\nLangChain simplifies every stage of the LLM application lifecycle:'), Document(id='b056df70-8b39-4072-811b-23a9dfe89df2', metadata={'source': 'https://python.langchain.com/docs/introduction/', 'title': 'Introduction | 🦜️🔗 LangChain', 'description': 'LangChain is a framework for developing applications powered by large language models (LLMs).', 'language': 'en'}, page_content="langchain-core: Base abstractions for chat models and other components.\nIntegration packages (e.g. langchain-openai, langchain-anthropic, etc.): Important integrations have been split into lightweight packages that 